<a href="https://colab.research.google.com/github/artuurog/3d_reconstruction_colmap/blob/main/colmap_3d_reconstruction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D reconstruction from images using COLMAP
## https://colmap.org/

In [ ]:
import subprocess, os

# check GPU availability
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU available")

# Install COLMAP
subprocess.run(['apt-get', 'install', '-y', 'colmap'],
               capture_output=True)

# check version
result = subprocess.run(['colmap', '--version'], capture_output=True, text=True)
print("COLMAP:", result.stdout.strip())

# Install Open3D for post-processing
subprocess.run(['pip', 'install', 'open3d', '-q'], capture_output=True)
print("Open3D installed.")

Mon May 11 14:56:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Wrapper for pipeline

In [ ]:
import subprocess
import os
from pathlib import Path

class ColmapPipeline:
    def __init__(self, workspace: str, images: str, use_gpu: bool = True):
        self.workspace = Path(workspace)
        self.images    = Path(images)
        self.db        = self.workspace / "database.db"
        self.sparse    = self.workspace / "sparse"
        self.dense     = self.workspace / "dense"
        self.use_gpu   = "1" if use_gpu else "0"

        # Crea cartelle
        self.sparse.mkdir(parents=True, exist_ok=True)
        self.dense.mkdir(parents=True, exist_ok=True)

    def _run(self, args: list, step: str):
        """Executes a COLMAP command and prints output."""
        print(f"\n{'='*50}")
        print(f"▶ {step}")
        print(f"{'='*50}")

        process = subprocess.Popen(
            args,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for line in process.stdout:
            print(line, end='')

        process.wait()
        if process.returncode != 0:
            raise RuntimeError(f"COLMAP failed at step: {step}")
        print(f"✅ {step} completed")

    def feature_extraction(self):
        self._run([
            'colmap', 'feature_extractor',
            '--database_path',             str(self.db),
            '--image_path',                str(self.images),
            '--SiftExtraction.use_gpu',    self.use_gpu,
            '--SiftExtraction.max_image_size', '3200',
            '--SiftExtraction.max_num_features', '8192',
            '--ImageReader.single_camera', '0',
        ], "Feature extraction (SIFT)")

    def feature_matching(self, method='exhaustive'):
        """
        method: 'exhaustive' (< ~300 img) | 'vocab_tree' (large dataset)
        """
        if method == 'vocab_tree':
            # Scarica vocab tree se non presente
            vt_path = '/content/vocab_tree_flickr100K_words32K.bin'
            if not os.path.exists(vt_path):
                print("Download vocab tree...")
                subprocess.run([
                    'wget', '-q', '-O', vt_path,
                    'https://demuc.de/colmap/vocab_tree_flickr100K_words32K.bin'
                ])
            args = [
                'colmap', 'vocab_tree_matcher',
                '--database_path',          str(self.db),
                '--SiftMatching.use_gpu',   self.use_gpu,
                '--VocabTreeMatching.vocab_tree_path', vt_path,
            ]
        else:
            args = [
                'colmap', 'exhaustive_matcher',
                '--database_path',        str(self.db),
                '--SiftMatching.use_gpu', self.use_gpu,
            ]
        self._run(args, f"Feature matching ({method})")

    def sparse_reconstruction(self):
        self._run([
            'colmap', 'mapper',
            '--database_path', str(self.db),
            '--image_path',    str(self.images),
            '--output_path',   str(self.sparse),
            '--Mapper.num_threads', '4',
            '--Mapper.init_min_tri_angle', '4',
            '--Mapper.multiple_models', '0',
        ], "SfM — Sparse reconstruction")

    def undistort(self):
        self._run([
            'colmap', 'image_undistorter',
            '--image_path',  str(self.images),
            '--input_path',  str(self.sparse / '0'),
            '--output_path', str(self.dense),
            '--output_type', 'COLMAP',
            '--max_image_size', '2000',
        ], "Undistort images")

    def dense_reconstruction(self, use_gpu_mvs: bool = True):
        if use_gpu_mvs:
            # PatchMatch Stereo (requires CUDA)
            self._run([
                'colmap', 'patch_match_stereo',
                '--workspace_path',   str(self.dense),
                '--workspace_format', 'COLMAP',
                '--PatchMatchStereo.geom_consistency', 'true',
                '--PatchMatchStereo.max_image_size',   '2000',
            ], "MVS — PatchMatch Stereo (GPU)")
        else:
            print("⚠️  No GPU: fallback to depth map fusion")

    def fuse_point_cloud(self):
        output_ply = self.dense / 'fused.ply'
        self._run([
            'colmap', 'stereo_fusion',
            '--workspace_path',   str(self.dense),
            '--workspace_format', 'COLMAP',
            '--input_type',       'geometric',
            '--output_path',      str(output_ply),
            '--StereoFusion.min_num_pixels', '3',
        ], "Stereo fusion — Dense point cloud")
        return output_ply

    def run_full_pipeline(self, matching='exhaustive'):
        self.feature_extraction()
        self.feature_matching(matching)
        self.sparse_reconstruction()
        self.undistort()
        self.dense_reconstruction()
        return self.fuse_point_cloud()

Upload images

In [ ]:
from google.colab import drive, files
import zipfile, shutil

# Option A: Google Drive
drive.mount('/content/drive')
SOURCE = '/content/drive/MyDrive/mie_immagini/'

# Opzione B: direct upload (uncomment)
# uploaded = files.upload()
# os.makedirs('/content/images', exist_ok=True)
# for fname in uploaded:
#     shutil.move(fname, f'/content/images/{fname}')
# SOURCE = '/content/images'

# Opzione C: import from zip
# with zipfile.ZipFile('/content/drive/MyDrive/scene.zip') as z:
#     z.extractall('/content/images')
# SOURCE = '/content/images'

Execute pipeline

In [ ]:
pipeline = ColmapPipeline(
    workspace='/content/sfm_project',
    images=SOURCE,
    use_gpu=True # false if no GPU avaiable
)

# For dataset < 300 images use 'exhaustive',
# for large datasets use 'vocab_tree'
fused_ply = pipeline.run_full_pipeline(matching='exhaustive')
print(f"\n🎉 Point cloud saved in: {fused_ply}")

Mesh with Poisson reconstruction

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.io.read_point_cloud(str(fused_ply))
print(f"Pointcloud: {len(pcd.points):,}")

# Estimate orthogonal parameters
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)
pcd.orient_normals_consistent_tangent_plane(100)

mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=9, width=0, scale=1.1, linear_fit=False
)

threshold = np.quantile(densities, 0.02)
mask = np.asarray(densities) < threshold
mesh.remove_vertices_by_mask(mask)
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()

mesh_path = '/content/sfm_project/output_mesh.ply'
o3d.io.write_triangle_mesh(mesh_path, mesh)
print(f"✅ Mesh saved: {mesh_path}")
print(f"   Vertices: {len(mesh.vertices):,}  |  Triangles: {len(mesh.triangles):,}")

Download results

In [ ]:
from google.colab import files
import zipfile

with zipfile.ZipFile('/content/results.zip', 'w') as z:
    z.write(str(fused_ply),  'fused.ply')        # dense point cloud
    z.write(mesh_path,       'output_mesh.ply')  # final mesh

files.download('/content/results.zip')